In [ ]:
%pip install langchain-experimental watermark

In [1]:
import pandas as pd
import os
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_nvidia_ai_endpoints import ChatNVIDIA

In [2]:
# 1. Load the Titanic dataset into a single DataFrame
url = "https://raw.githubusercontent.com/adamerose/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Let's peek at what the spreadsheet columns look like
print(df.head(2))

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  


In [3]:
# The correct, modern way to import
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os
from dotenv import load_dotenv


load_dotenv()

nvidia_key = os.getenv("NVIDIA_API_KEY")
# 2. Configure your Nvidia API Key

# # Initialize the model with 0.0 temperature for precise code calculation
llm = ChatNVIDIA(
    model="meta/llama-3.1-70b-instruct", 
    nvidia_api_key=nvidia_key,
    temperature=0.1,
)

In [ ]:
pip install tabulate

In [4]:
# # 3. Create the Agent for our SINGLE dataframe (df)
agent = create_pandas_dataframe_agent(
    llm,
    df, # We pass just ONE dataframe here instead of a list
    verbose=True,
    allow_dangerous_code=True # Allows the agent to run pandas operations locally
)

In [5]:
response = agent.invoke({"input": "how many passengers survived?"})



> Entering new AgentExecutor chain...
Thought: To find out how many passengers survived, I need to count the number of rows in the dataframe where the 'survived' column is 1.

Action: python_repl_ast
Action Input: df[df['survived'] == 1].shape[0]
342Action: python_repl_ast 
Action Input: df[df['survived'] == 1].shape[0] 342Final Answer: 342

> Finished chain.


In [6]:
response

{'input': 'how many passengers survived?', 'output': '342'}

In [8]:
df1 = df.copy()

df1["age"] = df1["age"].fillna(df1["age"].mean())



In [ ]:
agent1 = create_pandas_dataframe_agent(
    llm,
    [df, df1],
    verbose=True,
    allow_dangerous_code=True
)

In [ ]:
response = agent1.invoke({
    "input": "Compare the 'age' column in both dataframes. What is the difference in missing values?"
})
print(response["output"])



> Entering new AgentExecutor chain...
Thought: To compare the 'age' column in both dataframes and find the difference in missing values, I need to count the number of missing values in each dataframe and then subtract them.

Action: python_repl_ast
Action Input: df1['age'].isnull().sum() - df2['age'].isnull().sum()177The difference in missing values in the 'age' column between the two dataframes is 177. However, this does not tell us which dataframe has more missing values. To determine this, we need to find out which dataframe has a positive difference and which has a negative difference.

Action: python_repl_ast 
Action Input: df1['age'].isnull().sum(), df2['age'].isnull().sum() (np.int64(177), np.int64(0))Thought: Now that we have the number of missing values in each dataframe, we can determine which dataframe has more missing values. Since df1 has 177 missing values and df2 has 0 missing values, df1 has more missing values.

Final Answer: The difference in missing values in the '

In [12]:
df2=df1.copy()

In [13]:
df2["Age_Multiplied"]= df1["age"] * 2

In [14]:
df2.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,Age_Multiplied
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,44.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,76.0
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,52.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,70.0
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,70.0


In [15]:
agent2 = create_pandas_dataframe_agent(
    llm,
    [df, df1,df2],
    verbose=True,
    allow_dangerous_code=True
)

In [16]:
response = agent2.invoke({
    "input": "Are the numbers of column are same in all the dataframes?"
})
print(response)



> Entering new AgentExecutor chain...
Thought: To check if the number of columns are the same in all the dataframes, we can compare the length of the columns in each dataframe.

Action: python_repl_ast
Action Input: len(df1.columns) == len(df2.columns) == len(df3.columns)
FalseThe number of columns in the dataframes are not the same. To find out which dataframe has a different number of columns, we need to compare the lengths individually.

Action: python_repl_ast
Action Input: print(len(df1.columns), len(df2.columns), len(df3.columns))15 15 16
It seems that df3 has a different number of columns compared to df1 and df2. To find out which column is extra in df3, we can compare the columns of df1 and df3.

Action: python_repl_ast
Action Input: print(set(df3.columns) - set(df1.columns)){'Age_Multiplied'}
The extra column in df3 is 'Age_Multiplied'. 

Final Answer: No, the numbers of columns are not the same in all the dataframes. df3 has an extra column 'Age_Multiplied' compared to df1 

In [ ]:


# # 1. Initialize your Nvidia LLM
# os.environ["NVIDIA_API_KEY"] = "nvapi-your-key"
# llm = ChatNVIDIA(model="meta/llama-3.1-70b-instruct", temperature=0.0) 
# # Note: Temperature is set to 0.0 because we want precise code generation, not creativity!

# # 2. Sample Dataframes (e.g., for Pure Scan)
# df_products = pd.DataFrame({
#     "product_name": ["Alpha Chips", "Beta Soda", "Gamma Candy"],
#     "ingredient_code": ["E102", "E211", "E330"]
# })

# df_toxicity = pd.DataFrame({
#     "code": ["E102", "E211", "E330"],
#     "risk_level": ["High", "Medium", "Safe"],
#     "side_effect": ["Hyperactivity", "Allergies", "None"]
# })

# # 3. Create the Multi-DataFrame Agent
# # Pass BOTH dataframes in a list: [df1, df2]
# agent_executor = create_pandas_dataframe_agent(
#     llm,
#     [df_products, df_toxicity],
#     verbose=True,
#     allow_dangerous_code=True # Required in newer versions to let the agent run local pandas code
# )

# # 4. Test Query requiring cross-referencing
# response = agent_executor.invoke({
#     "input": "Which products contain ingredients with a High risk_level and what are their side effects?"
# })

# print(response["output"])